In [ ]:
import pandas as pd
import numpy as np

# URLs das métricas
urls = {
    'ARIMA': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Combinar/ARIMA_modelos_metricas_arima.csv',
    'FDM': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Combinar/FDM_metricas_por_sexo.csv',
    'LC': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Combinar/LC_metricas_por_sexo.csv',
    'ETS': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Combinar/ets_metricas.csv',
    'CNN_GRU': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Combinar/transfer_cnn_gru_metricas_validacao.csv',
    'CNN': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Combinar/transfer_cnn_metricas_validacao.csv',
    'GRU': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Combinar/transfer_gru_metricas_validacao.csv'
}

# Carregar e preparar os dados de cada modelo
all_data = []
for model_name, url in urls.items():
    df = pd.read_csv(url)
    # Manter apenas as colunas de interesse
    df = df[['sexo', 'idade', 'RMSE', 'MAE', 'sMAPE']]
    # Renomear colunas de métricas para incluir o nome do modelo
    df = df.rename(columns={
        'RMSE': f'RMSE_{model_name}',
        'MAE': f'MAE_{model_name}',
        'sMAPE': f'sMAPE_{model_name}'
    })
    all_data.append(df)

# Criar um DataFrame base com todos os pares únicos de sexo e idade
base_df = pd.concat(all_data)[['sexo', 'idade']].drop_duplicates().sort_values(['sexo', 'idade']).reset_index(drop=True)

# Juntar os dados de todos os modelos
df_final = base_df.copy()
for df_model in all_data:
    df_final = pd.merge(df_final, df_model, on=['sexo', 'idade'], how='left')

# Reordenar as colunas para o formato solicitado
model_names = list(urls.keys())
cols_ordered = ['sexo', 'idade']
for metrica in ['sMAPE', 'MAE', 'RMSE']:
    for model in model_names:
        cols_ordered.append(f'{metrica}_{model}')

df_final = df_final[cols_ordered]



In [ ]:
df_final

In [ ]:
# Altera as colunas *CNN_GRU para CNNGRU
df_final.rename(columns={
    'sMAPE_CNN_GRU': 'sMAPE_CNNGRU',
    'MAE_CNN_GRU': 'MAE_CNNGRU',
    'RMSE_CNN_GRU': 'RMSE_CNNGRU'
}, inplace=True)

In [ ]:
# Salvar em um arquivo CSV
df_final.to_csv('metricas_consolidadas_por_modelo.csv', index=False)

print("Arquivo 'metricas_consolidadas_por_modelo.csv' gerado com sucesso.")
print("\nAqui estão as primeiras linhas do DataFrame consolidado:")
print(df_final.head())

print("\nEstrutura do DataFrame:")
print(f"Linhas: {df_final.shape[0]}, Colunas: {df_final.shape[1]}")
print("\nNomes das colunas:")
print(df_final.columns.tolist())

In [ ]:
metricas_val = df_final.copy()

In [ ]:
import pandas as pd

# Supondo que 'metricas_val' seja seu DataFrame
metricas_long = (
    metricas_val
    .melt(id_vars=['idade', 'sexo'],
          var_name='METRICA_MODELO',
          value_name='VALOR')
    .assign(
        METRICA=lambda x: x['METRICA_MODELO'].str.split('_').str[0],
        MODELO=lambda x: x['METRICA_MODELO'].str.split('_').str[1]
    )
    .drop(columns=['METRICA_MODELO'])
    .pivot_table(index=['idade', 'sexo', 'MODELO'],
                 columns='METRICA',
                 values='VALOR')
    .reset_index()
    .rename_axis(columns=None)
)

# Calcular as métricas normalizadas e o score
metricas_long = (
    metricas_long
    .groupby(['idade', 'sexo'])
    .apply(lambda x: x.assign(
        sMAPE_norm=x['sMAPE'] / x['sMAPE'].max(),
        MAE_norm=x['MAE'] / x['MAE'].max(),
        RMSE_norm=x['RMSE'] / x['RMSE'].max()
    ))
    .reset_index(drop=True)
    .assign(score=lambda x: (x['sMAPE_norm'] + x['MAE_norm'] + x['RMSE_norm']) / 3)
)

In [ ]:
metricas_long.head(14)

In [ ]:
# Alternativa mais eficiente
metricas_peso = metricas_long[['idade', 'sexo', 'MODELO', 'score']].copy()
metricas_peso['peso'] = 1 / metricas_peso['score']

# Normalizar pesos por grupo
metricas_peso['peso_norm'] = (
    metricas_peso
    .groupby(['idade', 'sexo'])['peso']
    .transform(lambda x: x / x.sum())
)

In [ ]:
metricas_peso.head(14)

In [ ]:
import pandas as pd

# Carregar todas as previsões
urls = {
    'ARIMA': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Previs%C3%B5es/ARIMA_previsoes_intervalos_arima.csv',
    'ETS': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Previs%C3%B5es/ETS_previsoes.csv',
    'FDM': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Previs%C3%B5es/FDM_prev_total.csv',
    'LC': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Previs%C3%B5es/LC_prev_total.csv',
    'CNNGRU': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Previs%C3%B5es/previsoes_cnn_gru_2016_2019.csv',
    'CNN': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Previs%C3%B5es/transfer_cnn_previsoes.csv',
    'GRU': 'https://raw.githubusercontent.com/cleodecker/TCC/refs/heads/main/Previs%C3%B5es/transfer_gru_previsoes.csv'
}

# Carregar e consolidar as previsões em formato wide
previsoes_wide = None

for modelo, url in urls.items():
    df = pd.read_csv(url)
    df = df[['sexo', 'idade', 'ano', 'previsto']].copy()
    df = df.rename(columns={'previsto': modelo})

    if previsoes_wide is None:
        previsoes_wide = df
    else:
        previsoes_wide = previsoes_wide.merge(df, on=['sexo', 'idade', 'ano'], how='outer')

# Filtrar apenas para o período 2016-2019
previsoes_wide = previsoes_wide[(previsoes_wide['ano'] >= 2016) & (previsoes_wide['ano'] <= 2019)]

# Transformar os pesos para formato wide (uma coluna para cada modelo)
pesos_wide = metricas_peso.pivot_table(
    index=['sexo', 'idade'],
    columns='MODELO',
    values='peso_norm'
).reset_index()

# Renomear as colunas para ter o prefixo 'peso_'
pesos_wide.columns = ['peso_' + col if col != 'sexo' and col != 'idade' else col for col in pesos_wide.columns]

# Juntar as previsões com os pesos
dados_completos = previsoes_wide.merge(pesos_wide, on=['sexo', 'idade'], how='left')

# Calcular a previsão combinada
modelos = list(urls.keys())
dados_completos['previsao_combinada'] = 0

for modelo in modelos:
    # Verificar se a coluna do modelo e do peso existem
    if modelo in dados_completos.columns and f'peso_{modelo}' in dados_completos.columns:
        dados_completos['previsao_combinada'] += dados_completos[modelo] * dados_completos[f'peso_{modelo}']

# Adicionar os valores observados (pegando de qualquer um dos dataframes originais)
df_obs = pd.read_csv(urls['ARIMA'])[['sexo', 'idade', 'ano', 'observado']]
dados_completos = dados_completos.merge(df_obs, on=['sexo', 'idade', 'ano'], how='left')

# Selecionar apenas as colunas relevantes
resultado_final = dados_completos[['sexo', 'idade', 'ano', 'observado', 'previsao_combinada'] + modelos]

# Ordenar por sexo, ano e idade
resultado_final = resultado_final.sort_values(by=['sexo', 'ano', 'idade'])

print("Previsões combinadas calculadas:")
print(resultado_final.head(14))

In [ ]:
# Calcular métricas de erros sMAPE, MAE e RMSE entre previsao_combinada e observado
# Função para calcular sMAPE
def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

# Agrupar por 'sexo' e 'idade' e calcular as métricas
resultados = dados_completos.groupby(['sexo', 'idade']).apply(
    lambda x: pd.Series({
        'RMSE': np.sqrt(np.mean((x['previsao_combinada'] - x['observado']) ** 2)),
        'MAE': np.mean(np.abs(x['previsao_combinada'] - x['observado'])),
        'sMAPE': smape(x['observado'], x['previsao_combinada'])
    })
).reset_index()

print(resultados)

In [ ]:
# Média por sexo
resultados_sexo = resultados.groupby('sexo').mean()
print(resultados_sexo)
#

In [ ]:
# Salvar métricas e previsões
resultados.to_csv('metricas_previsoes_combinadas.csv', index=False)
resultado_final.to_csv('previsoes_combinadas.csv', index=False)